In [1]:
# ============================================================================
# 1. SETUP & CONFIGURATION
# ============================================================================
import os
import time
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pencilbox as pb

# ---- Reporting period --------------------------------------------------------
start_date = '2026-04-01'
end_date   = '2026-06-30'

# ---- Configuration ----
OUTPUT_DIR = os.path.expanduser('Data/')
os.makedirs(OUTPUT_DIR, exist_ok=True)

CHUNKSIZE = 100_000  # Number of rows to pull per batch (adjust based on memory constraints)

# ---- Connection ----
print("Establishing connection to Trino...")
# Uses the standard Pencilbox Trino connection from your environment
conn = pb.get_connection('[Warehouse] Trino')
print("Connection established successfully.")

Establishing connection to Trino...
Connection established successfully.


In [2]:
# ============================================================================
# 2. EXPORT ENGINE
# ============================================================================
def execute_and_export(query: str, filename: str, chunksize: int = CHUNKSIZE):
    """
    Executes a SQL query in chunks, displays a sample, and writes to a Parquet file.
    """
    output_path = os.path.join(OUTPUT_DIR, filename)
    print(f"Starting execution. Output will be saved to: {output_path}")
    
    start_time = time.time()
    writer = None
    total_rows = 0
    
    try:
        # Fetch data in chunks to prevent memory overload
        for i, chunk in enumerate(pd.read_sql_query(query, conn, chunksize=chunksize)):
            if len(chunk) == 0:
                continue
                
            # If it's the first chunk, display a sample and initialize the ParquetWriter
            if i == 0:
                print("\n--- Sample Data (First 5 Rows) ---")
                display(chunk.head())
                print("----------------------------------\n")
                
                # Convert first chunk to Arrow Table to infer schema
                table = pa.Table.from_pandas(chunk, preserve_index=False)
                writer = pq.ParquetWriter(output_path, table.schema)
            else:
                # Convert subsequent chunks
                table = pa.Table.from_pandas(chunk, preserve_index=False)
            
            # Write chunk to file
            writer.write_table(table)
            total_rows += len(chunk)
            print(f"Processed {total_rows:,} rows...")
            
    except Exception as e:
        print(f"An error occurred: {e}")
        
    finally:
        # Ensure the writer is properly closed even if an error occurs
        if writer:
            writer.close()
            
    elapsed_time = time.time() - start_time
    print(f"\nDone! Successfully exported {total_rows:,} rows in {elapsed_time:.1f} seconds.")

In [3]:
# ============================================================================
# 3. RUN YOUR QUERY
# ============================================================================

# Define your custom query here
MY_QUERY = """
 WITH mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names

    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
),
raw as (
SELECT
    c.id                               AS contract_id,
    c.state                            AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,

    c.source_company_id              AS company_purchaser_id,
    c.source_company_name            AS company_purchaser_name,
    c.destination_company_id                  AS company_buyer_id,
    c.destination_company_name                AS company_buyer_name,
    s.name                             AS section_name,
    qa.id                              AS question_answer_id,
    qa.question_id,
    -- qa.answer,
    json_parse(qa.answer) as answer,
    qa.remarks,
    qa.created_at                      AS qa_created_at,
    qa.created_by                      AS qa_created_by,
    ma.mappings,
    ma.manufacturer_ids,
    ma.manufacturer_names,
    c.execution_date,start_date as effective_date,
    case when c.end_date is null then 'Yes' else 'No' end as valid_until_cancelled,
c.updated_at as contract_updated_at
FROM clm.clm_contract c
LEFT JOIN clm.clm_contract_section cs
       ON cs.contract_id = c.id
      AND cs.is_deleted = 0
LEFT JOIN clm.clm_section s
       ON s.id = cs.section_id
      AND s.is_deleted = 0
LEFT JOIN clm.clm_section_question_answer qa
       ON qa.contract_section_id = cs.id
      AND qa.is_deleted = 0
LEFT JOIN mapping_agg ma
       ON ma.contract_id = c.id
WHERE c.is_deleted = 0
and c.state not in ('ARCHIVED','REJECTED')
ORDER BY
    cs.section_id NULLS LAST,
    qa.question_id NULLS LAST,
    qa.id NULLS LAST

),

format_ as (
select contract_id,
case when json_extract_scalar(x, '$.Manufacturer.id') is not null then 
json_extract_scalar(x, '$.Manufacturer.id') else cast(manufacturer_ids[1] as varchar) end
AS manufacturer_id,
case when json_extract_scalar(x, '$.Manufacturer.name') is not null then
json_extract_scalar(x, '$.Manufacturer.name') else cast(manufacturer_names[1] as varchar) end
AS manufacturer,
-- company_purchaser_id,
-- company_purchaser_name,
company_buyer_id,
company_buyer_name,
contract_state,
created_at,
created_by,
updated_at,
updated_by,
execution_date,
effective_date,
valid_until_cancelled,
section_name,
question_id,
question_answer_id,
contract_updated_at,
json_extract_scalar(x,'$.value') as answer_values
from
raw
cross join unnest (CASE 
        WHEN json_extract(answer, '$.values') IS NOT NULL 
        THEN CAST(json_extract(answer, '$.values') AS array(json))
        ELSE ARRAY[answer]
    END 
) as t(x)
),

fff as (
select contract_id,
manufacturer_id,
manufacturer,
-- company_buyer_id,
-- company_buyer_name,
contract_state,
date(execution_date) as execution_date,
date(effective_date) as effective_date,
-- date(contract_updated_at) as updated_at,/
valid_until_cancelled,
min(answer_values) filter(where question_id='kam_support_enable_v1') as kam_support,
min(answer_values) filter(where question_id='kam_charges_rs_v1') as kam_charges


from format_
group by 1,2,3,4,5,6,7--,8,9

)

select * from fff where kam_support='Yes'
"""

print(MY_QUERY)

# Name your output file (it will be saved in the OUTPUT_DIR defined in Cell 1)
OUTPUT_FILENAME = "KAM_Fees.parquet"

# Run the function
execute_and_export(query=MY_QUERY, filename=OUTPUT_FILENAME)


 WITH mapping_agg AS (
    SELECT
        cm.contract_id,
        ARRAY_AGG(
            CAST(
                ROW(cm.mapping_type, cm.id)
                AS ROW(mapping_type VARCHAR, ref_id BIGINT)
            )
        ) FILTER (WHERE cm.id IS NOT NULL) AS mappings,
        ARRAY_AGG(cm.mapped_id)
            FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_ids,
        ARRAY_AGG(
            cm.mapped_value
        ) FILTER (WHERE cm.mapping_type = 'MANUFACTURER') AS manufacturer_names

    FROM clm.clm_contract_mapping cm
    WHERE cm.is_deleted = 0
    GROUP BY cm.contract_id
),
raw as (
SELECT
    c.id                               AS contract_id,
    c.state                            AS contract_state,
    c.created_at,
    c.created_by,
    c.updated_at,
    c.updated_by,

    c.source_company_id              AS company_purchaser_id,
    c.source_company_name            AS company_purchaser_name,
    c.destination_company_id                  AS company_buyer_id

,contract_id,manufacturer_id,manufacturer,contract_state,execution_date,effective_date,valid_until_cancelled,kam_support,kam_charges
0,325,2568,Rama Vision Ltd.,PENDING APPROVAL,2025-11-16,2025-11-16,Yes,Yes,50000
1,1811,9423,Wellspring Consumer Pvt. Ltd. (formerly Kay Co...,PENDING APPROVAL,2026-02-12,2026-02-12,Yes,Yes,50000
2,610,34001,Helios Global Private Limited,PENDING APPROVAL,2025-10-31,2025-10-31,Yes,Yes,50000
3,636,2956,Connedit Business Solutions Pvt Ltd,PENDING APPROVAL,2026-01-11,2026-01-11,Yes,Yes,50000
4,243,3238,Groom Town Private Limited,APPROVED,2025-07-31,2025-07-31,Yes,Yes,50000


----------------------------------

Processed 169 rows...

Done! Successfully exported 169 rows in 55.6 seconds.
